# Huấn luyện 3D Gaussian Splatting (Train from scratch)

Notebook này thực hiện huấn luyện các scene của tập `private_set1` từ bộ dữ liệu đã được tiền xử lý (converted).

In [ ]:
# 1. Clone và cài đặt 3DGS (hoặc 2DGS tuỳ bạn chọn nhánh nào)
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd gaussian-splatting
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn

In [ ]:
import os
import subprocess

# 2. Cấu hình đường dẫn và scene cần train
DATASET_ROOT = '/kaggle/input/converted-bts-dataset/private_set1'
OUTPUT_ROOT = '/kaggle/working/outputs'

# Danh sách scene cần train. Để None hoặc [] nếu muốn train TẤT CẢ các scene trong folder
TARGET_SCENES = ['HCM0204']  # Ví dụ: ['HCM0204', 'HCM0181']

# Các tham số training
ITERATIONS = 15000

if not os.path.exists(DATASET_ROOT):
    print(f"Thư mục {DATASET_ROOT} không tồn tại. Vui lòng kiểm tra lại data input!")
else:
    all_scenes = sorted(os.listdir(DATASET_ROOT))
    
    # Lọc scene theo TARGET_SCENES
    if TARGET_SCENES:
        scenes = [s for s in all_scenes if s in TARGET_SCENES]
    else:
        scenes = all_scenes
        
    print(f"Tìm thấy {len(all_scenes)} scenes trong dataset.")
    print(f"Sẽ thực hiện train {len(scenes)} scenes: {scenes}")
    
    for scene in scenes:
        print("\n" + "="*60)
        print(f"TRAINING SCENE: {scene}")
        print("="*60)
        
        scene_path = os.path.join(DATASET_ROOT, scene, 'train')
        output_dir = os.path.join(OUTPUT_ROOT, scene)
        
        command = [
            "python", "train.py",
            "-s", scene_path,
            "-m", output_dir,
            "--eval",
            "--iterations", str(ITERATIONS),
            "--test_iterations", "7000", str(ITERATIONS),
            "--save_iterations", str(ITERATIONS),
            "--checkpoint_iterations", str(ITERATIONS),
            "--position_lr_max_steps", str(ITERATIONS),
            "--disable_viewer"
        ]
        
        # Chạy lệnh train và in log trực tiếp
        subprocess.run(command, check=True)
        print(f"Hoàn thành train 3DGS cho scene {scene}!")